# 42 — Adaptive Investor Outreach

Persona-adaptive generation with multi-model claim validation: builds a 3-year financial model, generates tailored fundraising materials per investor persona (VC / PE / family office), then cross-validates every key financial claim across 3 models before finalising. Two layers of nested parallelism keep wall-clock time low even as the number of LLM calls scales — up to 18 concurrent requests at peak.

## Framework Comparison

The persona-adaptive generation + multi-model claim validation pattern used here has direct equivalents in the major agent frameworks:

| Framework | Equivalent Pattern |
|-----------|-------------------|
| **LangGraph** | Multi-agent graph with a financial-model node feeding parallel persona-writer nodes, then parallel validator nodes with a flag-accumulator edge |
| **LangChain** | `LLMRouterChain` + `LLMCheckerChain` with a custom multi-LLM routing wrapper per claim |
| **CrewAI** | Multi-crew setup: FinanceAnalyst crew -> PersonaWriter crew (3 agents) -> FactChecker crew (3 agents, one per validation model) |

This example implements the same logic directly with the OpenAI SDK and `concurrent.futures.ThreadPoolExecutor`, avoiding framework overhead while preserving full observability.

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."       # replace with your key
os.environ["OPENROUTER_API_KEY"] = "sk-or-..." # replace with your key (free at openrouter.ai)

In [ ]:
# ruff: noqa: E402
"""
Pydantic models for the adaptive investor outreach example.

Covers company input, financial projections, per-persona materials,
and multi-model claim validation results.
"""
from typing import Literal

from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# Input
# ---------------------------------------------------------------------------

InvestorPersona = Literal["vc", "pe", "family_office"]


class CompanyBrief(BaseModel):
    company_name: str = Field(description="Legal or trading name of the company.")
    sector: str = Field(description="Industry sector (e.g. 'B2B SaaS', 'Industrial Tech').")
    stage: Literal["seed", "series_a", "series_b", "growth"] = Field(
        description="Current funding stage."
    )
    arr_usd: float = Field(description="Current annual recurring revenue in USD.")
    growth_rate_pct: float = Field(
        description="Trailing ARR growth rate as a percentage (e.g. 80.0 for 80%)."
    )
    ebitda_margin_pct: float = Field(
        description="Current EBITDA margin as a percentage; negative values indicate a loss."
    )
    headcount: int = Field(description="Total full-time employee count.")
    key_differentiator: str = Field(
        description="One to two sentences describing the primary competitive moat."
    )
    use_of_funds: str = Field(
        description="Intended use of the capital being raised (e.g. 'GTM expansion, R&D hiring')."
    )


# ---------------------------------------------------------------------------
# Financial projection
# ---------------------------------------------------------------------------


class FinancialProjection(BaseModel):
    year1_arr: float = Field(description="Projected ARR at end of Year 1 in USD.")
    year2_arr: float = Field(description="Projected ARR at end of Year 2 in USD.")
    year3_arr: float = Field(description="Projected ARR at end of Year 3 in USD.")
    year3_ebitda: float = Field(
        description="Projected EBITDA at end of Year 3 in USD (may be negative)."
    )
    implied_valuation_usd: float = Field(
        description="Implied enterprise valuation derived from Year 3 ARR or EBITDA multiples."
    )
    key_assumptions: list[str] = Field(
        description="3-6 modelling assumptions underpinning the projections. "
        "Flag high-uncertainty items with a '(!)' prefix."
    )


# ---------------------------------------------------------------------------
# Investor materials (per persona)
# ---------------------------------------------------------------------------


class InvestorMaterials(BaseModel):
    persona: InvestorPersona = Field(
        description="Target investor persona: vc, pe, or family_office."
    )
    pitch_angle: str = Field(
        description="One to two sentence framing of why this investment fits the persona's mandate."
    )
    key_metrics_highlighted: list[str] = Field(
        description="3-5 metrics most likely to resonate with this persona (e.g. NRR, EBITDA, IRR)."
    )
    narrative_hook: str = Field(
        description="Opening hook for the outreach -- tailored to the persona's language and priorities."
    )
    risk_framing: str = Field(
        description="How key risks should be presented to this audience to build trust rather than alarm."
    )
    ask_structure: str = Field(
        description="How to frame the raise (ticket size, instrument, timeline) for this persona."
    )


# ---------------------------------------------------------------------------
# Claim validation
# ---------------------------------------------------------------------------


class ClaimValidation(BaseModel):
    claim: str = Field(description="The exact financial claim being assessed.")
    model_used: str = Field(
        description="The model that produced this validation (e.g. openai/gpt-4o-mini)."
    )
    verdict: Literal["confirmed", "disputed", "inconclusive"] = Field(
        description="Whether the model considers the claim defensible given the stated assumptions."
    )
    confidence: float = Field(
        description="Model's self-assessed confidence from 0.0 (none) to 1.0 (certain)."
    )
    note: str = Field(
        description="Brief rationale for the verdict, including any caveats or flagged risks."
    )


class ValidatedClaim(BaseModel):
    claim: str = Field(description="The financial claim that was validated.")
    validated: bool = Field(
        description="True when fewer than 2 of the 3 validation models dispute the claim."
    )
    supporting_validations: list[ClaimValidation] = Field(
        description="One ClaimValidation entry per model queried."
    )
    dissenting_count: int = Field(
        description="Number of models that returned a 'disputed' verdict."
    )


# ---------------------------------------------------------------------------
# Final output
# ---------------------------------------------------------------------------


class OutreachPackage(BaseModel):
    company_name: str = Field(description="Name of the company this package was built for.")
    projection: FinancialProjection = Field(
        description="The 3-year financial projection underpinning all materials."
    )
    materials: list[InvestorMaterials] = Field(
        description="One InvestorMaterials object for each of the three personas."
    )
    validated_claims: list[ValidatedClaim] = Field(
        description="Each key financial claim with its multi-model validation result."
    )
    flagged_claims: list[str] = Field(
        description="Claims where 2 or more validation models returned a 'disputed' verdict."
    )

In [ ]:
"""
System prompt constants for the adaptive investor outreach example.

Three prompts cover:
  - PROJECTION_SYSTEM  : build a conservative 3-year financial model
  - MATERIALS_SYSTEM   : write persona-tailored investor materials
  - VALIDATE_SYSTEM    : cross-validate a single financial claim
"""

PROJECTION_SYSTEM = (
    "You are a senior financial analyst building a conservative 3-year revenue model "
    "for an early-stage company.\n\n"
    "Given a company brief, produce a FinancialProjection that:\n"
    "- Extrapolates ARR over Years 1-3 using the provided growth rate, but applies "
    "  a realistic deceleration curve (growth typically slows as the base scales).\n"
    "- Estimates Year-3 EBITDA by modelling operating leverage against headcount and "
    "  stage-appropriate burn assumptions.\n"
    "- Derives an implied valuation using a sector-appropriate ARR or EBITDA multiple "
    "  (be explicit about which multiple you used).\n"
    "- Lists 3-6 key modelling assumptions. Prefix any high-uncertainty assumption "
    "  with '(!)' so readers can assess sensitivity.\n\n"
    "Bias towards conservative estimates. Do not project hockey-stick growth unless "
    "the growth rate and sector data strongly support it. If an assumption carries "
    "material uncertainty, flag it."
)

MATERIALS_SYSTEM = (
    "You are an experienced investor relations advisor preparing outreach materials "
    "for a fundraising round.\n\n"
    "Given a company brief, a 3-year financial projection, and a target investor persona "
    "(vc | pe | family_office), produce InvestorMaterials that:\n\n"
    "VC persona:\n"
    "  - Lead with TAM and growth velocity. Emphasise ARR growth rate, NRR, and "
    "    path to market leadership.\n"
    "  - Frame risks around execution speed and competitive moats.\n"
    "  - Structure the ask as a priced equity round with standard VC terms.\n\n"
    "PE persona:\n"
    "  - Lead with EBITDA trajectory and free cash flow potential. Emphasise "
    "    operational efficiency, margin expansion, and LBO/growth equity fit.\n"
    "  - Frame risks around integration complexity and defensibility of margins.\n"
    "  - Structure the ask around enterprise value, leverage ratios, and exit multiples.\n\n"
    "family_office persona:\n"
    "  - Lead with capital preservation and long-term compounding. Emphasise "
    "    downside protection, diversification benefits, and yield characteristics.\n"
    "  - Frame risks in terms of time horizon and illiquidity premium.\n"
    "  - Structure the ask with flexible instruments (convertible notes, preferred equity).\n\n"
    "All materials must be direct, jargon-appropriate for the audience, and grounded "
    "in the numbers in the projection. Do not invent metrics not in the brief."
)

VALIDATE_SYSTEM = (
    "You are an independent financial analyst reviewing a single financial claim made "
    "in investor materials.\n\n"
    "Given:\n"
    "  - The claim (e.g. 'ARR will grow from $2M to $8M in 3 years')\n"
    "  - The modelling assumptions that underpin the projection\n\n"
    "Assess whether the claim is defensible:\n"
    "  - 'confirmed'    : the claim follows logically from the assumptions and is "
    "    consistent with comparable company benchmarks.\n"
    "  - 'disputed'     : the claim is materially inconsistent with the assumptions, "
    "    overstates what the data supports, or contradicts sector norms.\n"
    "  - 'inconclusive' : there is insufficient information to confirm or dispute; "
    "    the claim is plausible but unverifiable from the provided context.\n\n"
    "Provide a confidence score from 0.0 (no confidence) to 1.0 (high confidence) in "
    "your verdict, and a brief note (1-3 sentences) explaining your reasoning.\n\n"
    "Be rigorous. Do not default to 'confirmed' simply because the numbers look "
    "internally consistent -- assess against real-world sector benchmarks."
)

In [ ]:
# ruff: noqa: E402
"""
Multi-model claim validation via OpenRouter.

Mirrors the fan-out pattern from example 27 (multi-provider-fan-out):
each model independently assesses whether a financial claim is defensible
given the stated projection assumptions.
"""
import os

from openai import OpenAI

VALIDATION_MODELS = [
    "openai/gpt-4o-mini",
    "mistralai/mistral-7b-instruct",
    "meta-llama/llama-3.1-8b-instruct",
]


def _get_client() -> OpenAI:
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )


def validate_claim_with_model(
    claim: str,
    assumptions: list[str],
    model: str,
) -> ClaimValidation:
    """Ask a single model to validate one financial claim.

    Args:
        claim: The financial claim to assess (e.g. 'ARR grows from $2M to $8M').
        assumptions: The key modelling assumptions from the FinancialProjection.
        model: OpenRouter model string (e.g. 'openai/gpt-4o-mini').

    Returns:
        A ClaimValidation with verdict, confidence, and note from that model.
    """
    client = _get_client()
    assumptions_text = "\n".join(f"- {a}" for a in assumptions)
    user_message = (
        f"Claim to validate:\n{claim}\n\n"
        f"Modelling assumptions:\n{assumptions_text}"
    )
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": VALIDATE_SYSTEM},
            {"role": "user", "content": user_message},
        ],
        response_format=ClaimValidation,
    )
    result: ClaimValidation = completion.choices[0].message.parsed
    result.claim = claim
    result.model_used = model
    return result

In [ ]:
# ruff: noqa: E402
"""
Adaptive investor outreach workflow.

Orchestrates three parallel concerns:
  1. Financial modelling   -- build a conservative 3-year projection
  2. Persona generation    -- fan out InvestorMaterials for VC / PE / family_office
  3. Claim validation      -- fan out each key claim across 3 validation models

All LLM calls that can run in parallel do so via ThreadPoolExecutor.
"""
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import get_args

from openai import OpenAI

_PERSONAS: list[InvestorPersona] = list(get_args(InvestorPersona))

_PRIMARY_MODEL = "gpt-4o-mini"


def _get_primary_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# ---------------------------------------------------------------------------
# Step 1: financial projection
# ---------------------------------------------------------------------------


def _build_projection(brief: CompanyBrief) -> FinancialProjection:
    """Generate a conservative 3-year financial projection from a company brief."""
    client = _get_primary_client()
    brief_text = (
        f"Company: {brief.company_name}\n"
        f"Sector: {brief.sector}\n"
        f"Stage: {brief.stage}\n"
        f"Current ARR: ${brief.arr_usd:,.0f}\n"
        f"Growth rate: {brief.growth_rate_pct}% YoY\n"
        f"EBITDA margin: {brief.ebitda_margin_pct}%\n"
        f"Headcount: {brief.headcount}\n"
        f"Key differentiator: {brief.key_differentiator}\n"
        f"Use of funds: {brief.use_of_funds}"
    )
    completion = client.beta.chat.completions.parse(
        model=_PRIMARY_MODEL,
        messages=[
            {"role": "system", "content": PROJECTION_SYSTEM},
            {"role": "user", "content": brief_text},
        ],
        response_format=FinancialProjection,
    )
    return completion.choices[0].message.parsed


# ---------------------------------------------------------------------------
# Step 2: per-persona investor materials
# ---------------------------------------------------------------------------


def _build_materials(
    brief: CompanyBrief,
    projection: FinancialProjection,
    persona: InvestorPersona,
) -> InvestorMaterials:
    """Generate investor materials tailored to a specific persona."""
    client = _get_primary_client()
    projection_text = (
        f"Year 1 ARR: ${projection.year1_arr:,.0f}\n"
        f"Year 2 ARR: ${projection.year2_arr:,.0f}\n"
        f"Year 3 ARR: ${projection.year3_arr:,.0f}\n"
        f"Year 3 EBITDA: ${projection.year3_ebitda:,.0f}\n"
        f"Implied valuation: ${projection.implied_valuation_usd:,.0f}\n"
        f"Key assumptions:\n" + "\n".join(f"  - {a}" for a in projection.key_assumptions)
    )
    user_message = (
        f"Company brief:\n{brief.model_dump_json(indent=2)}\n\n"
        f"3-year projection:\n{projection_text}\n\n"
        f"Target persona: {persona}"
    )
    completion = client.beta.chat.completions.parse(
        model=_PRIMARY_MODEL,
        messages=[
            {"role": "system", "content": MATERIALS_SYSTEM},
            {"role": "user", "content": user_message},
        ],
        response_format=InvestorMaterials,
    )
    result: InvestorMaterials = completion.choices[0].message.parsed
    result.persona = persona
    return result


# ---------------------------------------------------------------------------
# Step 3: claim extraction and multi-model validation
# ---------------------------------------------------------------------------


def _extract_key_claims(projection: FinancialProjection) -> list[str]:
    """Derive the 3-5 most important numeric claims from the projection."""
    return [
        f"ARR will reach ${projection.year1_arr:,.0f} by end of Year 1.",
        f"ARR will reach ${projection.year2_arr:,.0f} by end of Year 2.",
        f"ARR will reach ${projection.year3_arr:,.0f} by end of Year 3.",
        f"EBITDA will reach ${projection.year3_ebitda:,.0f} by Year 3.",
        f"The implied enterprise valuation is ${projection.implied_valuation_usd:,.0f}.",
    ]


def _validate_claim(claim: str, projection: FinancialProjection) -> ValidatedClaim:
    """Fan out claim validation across all VALIDATION_MODELS in parallel."""
    assumptions = projection.key_assumptions
    validations = []

    with ThreadPoolExecutor(max_workers=len(VALIDATION_MODELS)) as executor:
        futures = {
            executor.submit(validate_claim_with_model, claim, assumptions, model): model
            for model in VALIDATION_MODELS
        }
        for future in as_completed(futures):
            validations.append(future.result())

    validations.sort(key=lambda v: VALIDATION_MODELS.index(v.model_used))
    dissenting = sum(1 for v in validations if v.verdict == "disputed")
    return ValidatedClaim(
        claim=claim,
        validated=dissenting < 2,
        supporting_validations=validations,
        dissenting_count=dissenting,
    )


# ---------------------------------------------------------------------------
# Public entrypoint
# ---------------------------------------------------------------------------


def run(brief: CompanyBrief) -> OutreachPackage:
    """Build a complete investor outreach package for all three personas.

    Steps:
      1. Generate a 3-year financial projection (sequential -- all else depends on it).
      2. Fan out persona materials for VC, PE, and family_office in parallel.
      3. Extract key claims from the projection.
      4. Fan out claim validation across 3 models for each claim, in parallel.
      5. Flag claims where 2+ models returned 'disputed'.

    Args:
        brief: A CompanyBrief describing the company seeking investment.

    Returns:
        An OutreachPackage containing the projection, per-persona materials,
        validated claims, and a list of flagged (disputed) claims.
    """
    # Step 1: projection (must complete before steps 2-4)
    projection = _build_projection(brief)

    # Step 2: persona materials -- fan out across all personas in parallel
    materials: list[InvestorMaterials] = []
    with ThreadPoolExecutor(max_workers=len(_PERSONAS)) as executor:
        persona_futures = {
            executor.submit(_build_materials, brief, projection, persona): persona
            for persona in _PERSONAS
        }
        for future in as_completed(persona_futures):
            materials.append(future.result())

    materials.sort(key=lambda m: _PERSONAS.index(m.persona))

    # Step 3: extract claims
    claims = _extract_key_claims(projection)

    # Step 4: validate each claim (each internally fans out to 3 models)
    validated_claims: list[ValidatedClaim] = []
    with ThreadPoolExecutor(max_workers=len(claims)) as executor:
        claim_futures = {
            executor.submit(_validate_claim, claim, projection): claim
            for claim in claims
        }
        for future in as_completed(claim_futures):
            validated_claims.append(future.result())

    validated_claims.sort(key=lambda vc: claims.index(vc.claim))

    # Step 5: collect flagged claims
    flagged = [vc.claim for vc in validated_claims if vc.dissenting_count >= 2]

    return OutreachPackage(
        company_name=brief.company_name,
        projection=projection,
        materials=materials,
        validated_claims=validated_claims,
        flagged_claims=flagged,
    )

In [ ]:
# ---------------------------------------------------------------------------
# Scenario A: B2B SaaS -- Series A (ClearPath AI)
# ---------------------------------------------------------------------------

SCENARIO_A = CompanyBrief(
    company_name="ClearPath AI",
    sector="B2B SaaS",
    stage="series_a",
    arr_usd=2_000_000,
    growth_rate_pct=80.0,
    ebitda_margin_pct=-35.0,
    headcount=28,
    key_differentiator=(
        "Proprietary NLP layer trained on 10M+ support tickets enables "
        "90-second ticket resolution vs. industry average of 8 minutes; "
        "net revenue retention of 128% after 18 months live."
    ),
    use_of_funds="GTM expansion (AEs, SDRs), product engineering (AI roadmap), 18-month runway.",
)


def _print_package(package) -> None:
    """Print a concise summary of an OutreachPackage."""
    print(f"\n{'=' * 70}")
    print(f"OUTREACH PACKAGE: {package.company_name}")
    print(f"{'=' * 70}")

    proj = package.projection
    print("\n[Financial Projection]")
    print(f"  Year 1 ARR   : ${proj.year1_arr:>14,.0f}")
    print(f"  Year 2 ARR   : ${proj.year2_arr:>14,.0f}")
    print(f"  Year 3 ARR   : ${proj.year3_arr:>14,.0f}")
    print(f"  Year 3 EBITDA: ${proj.year3_ebitda:>14,.0f}")
    print(f"  Implied EV   : ${proj.implied_valuation_usd:>14,.0f}")
    print("  Assumptions:")
    for assumption in proj.key_assumptions:
        print(f"    - {assumption}")

    print("\n[Investor Materials]")
    for mat in package.materials:
        print(f"\n  -- {mat.persona.upper()} --")
        print(f"  Pitch angle : {mat.pitch_angle}")
        print(f"  Hook        : {mat.narrative_hook}")
        print(f"  Key metrics : {', '.join(mat.key_metrics_highlighted)}")
        print(f"  Risk framing: {mat.risk_framing}")
        print(f"  Ask         : {mat.ask_structure}")

    print("\n[Claim Validation]")
    for vc in package.validated_claims:
        status = "PASS" if vc.validated else "FLAG"
        print(f"\n  [{status}] {vc.claim}")
        print(f"  Dissenters: {vc.dissenting_count} / {len(vc.supporting_validations)}")
        for cv in vc.supporting_validations:
            print(f"    {cv.model_used:<45} {cv.verdict:<14} conf={cv.confidence:.2f}")

    if package.flagged_claims:
        print("\n[Flagged Claims -- 2+ models disputed]")
        for claim in package.flagged_claims:
            print(f"  ! {claim}")
    else:
        print("\n[No claims flagged -- all passed multi-model validation]")


print(f"Processing: {SCENARIO_A.company_name} ({SCENARIO_A.stage}, {SCENARIO_A.sector}) ...")
package_a = run(SCENARIO_A)
_print_package(package_a)
print(f"\nFlagged claims: {package_a.flagged_claims}")

In [ ]:
# ---------------------------------------------------------------------------
# Scenario B: Industrial Tech -- Growth / PE-ready (IronBridge Dynamics)
# ---------------------------------------------------------------------------

SCENARIO_B = CompanyBrief(
    company_name="IronBridge Dynamics",
    sector="Industrial Tech",
    stage="growth",
    arr_usd=18_000_000,
    growth_rate_pct=32.0,
    ebitda_margin_pct=14.0,
    headcount=120,
    key_differentiator=(
        "Sensor-fusion platform for predictive maintenance in heavy manufacturing "
        "reduces unplanned downtime by 60%; 5-year contracts with top-10 auto OEMs "
        "provide highly visible, recurring revenue."
    ),
    use_of_funds="International expansion (EU & SE Asia), M&A pipeline for adjacent sensor data assets.",
)

print(f"Processing: {SCENARIO_B.company_name} ({SCENARIO_B.stage}, {SCENARIO_B.sector}) ...")
package_b = run(SCENARIO_B)
_print_package(package_b)
print(f"\nFlagged claims: {package_b.flagged_claims}")

## Starter Exercise

Add a **fourth investor persona**: `corporate_vc`.

Corporate VCs prioritise strategic fit, IP ownership, and technology exclusivity over pure financial returns.

To complete this exercise:

1. Add the new literal to `InvestorPersona` so it reads `Literal["vc", "pe", "family_office", "corporate_vc"]`.
2. Add persona-specific instructions to `MATERIALS_SYSTEM` covering what a corporate VC cares about: strategic alignment with the parent company's roadmap, co-development opportunities, and preferred exclusivity or right-of-first-refusal structures.
3. Verify the fan-out includes `corporate_vc` — because `_PERSONAS` is derived from `get_args(InvestorPersona)`, adding the literal is sufficient to route the new persona through the parallel `_build_materials` loop automatically.

Run Scenario A again after your changes and confirm a fourth materials block appears in the output.

In [ ]:
# ---------------------------------------------------------------------------
# Answer key: adding the corporate_vc persona
# ---------------------------------------------------------------------------

# 1. Updated InvestorPersona literal -- add "corporate_vc" as a fourth option
from typing import Literal, get_args

InvestorPersona = Literal["vc", "pe", "family_office", "corporate_vc"]

# 2. Extended MATERIALS_SYSTEM prompt -- append corporate_vc instructions
MATERIALS_SYSTEM_WITH_CORP_VC = (
    MATERIALS_SYSTEM  # reuse the existing prompt text defined in the prompts cell above
    + "\n\n"
    "corporate_vc persona:\n"
    "  - Lead with strategic fit: how does this technology accelerate or protect "
    "    the parent corporation's core business or R&D roadmap?\n"
    "  - Emphasise IP ownership potential, co-development rights, and any "
    "    technology exclusivity or right-of-first-refusal the deal could include.\n"
    "  - Frame risks around integration complexity with the parent's existing stack "
    "    and the risk of a competing portfolio company building similar capabilities.\n"
    "  - Structure the ask as a minority equity stake with a co-development side "
    "    letter and a defined path to deeper commercial partnership or acquisition."
)

# 3. _PERSONAS is re-derived automatically from the updated literal --
#    no manual list edit needed; the fan-out picks up corporate_vc immediately.
_PERSONAS_WITH_CORP_VC: list[InvestorPersona] = list(get_args(InvestorPersona))
print("Personas now included in fan-out:", _PERSONAS_WITH_CORP_VC)

# Patch globals so the existing run() and _build_materials() pick up the new values:
_PERSONAS = _PERSONAS_WITH_CORP_VC
MATERIALS_SYSTEM = MATERIALS_SYSTEM_WITH_CORP_VC  # noqa: F811 (intentional override)

print("\nRunning Scenario A with corporate_vc persona added ...")
package_a_extended = run(SCENARIO_A)
_print_package(package_a_extended)